In [1]:
!pip install google-adk --quiet!pip install -U google-adk!pip install httpx!pip install google-agents!pip install --upgrade google-generativeai[adk]!pip install google-adk -q!pip install litellm -q!pip install gradio --quiet

ERROR: Could not find a version that satisfies the requirement google-agents (from versions: none)
ERROR: No matching distribution found for google-agents
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 82.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires grpcio>=1.71.2, but you have grpcio 1.67.1 which is incompatible.


In [2]:
import osfrom google.adk.models.lite_llm import LiteLlmfrom google.adk.runners import Runnerfrom google.genai import typesos.environ["GROQ_API_KEY"] = "add_your_api_key"os.environ["GROQ_URL"] = "https://api.groq.com/openai/v1/chat/completions"os.environ["GROQ_MODEL"] = "llama3-8b-8192"lite_llm = LiteLlm(model="groq/llama3-8b-8192")

In [3]:
from google.adk.sessions import InMemorySessionServiceAPP_NAME = "real_estate_ai"USER_ID = "user_real_estate"SESSION_ID = "real_estate_session_001"initial_state = {    "user_preferences": {},    "search_results": [],    "scored_results": [],    "final_summary": ""}session_service = InMemorySessionService()session = await session_service.create_session(    app_name=APP_NAME,    user_id=USER_ID,    session_id=SESSION_ID,    state=initial_state)print(f" Session '{SESSION_ID}' created for user '{USER_ID}' with initial state:")print(session.state)

 Session 'real_estate_session_001' created for user 'user_real_estate' with initial state:
{'user_preferences': {}, 'search_results': [], 'scored_results': [], 'final_summary': ''}


In [4]:
from google.adk.agents import Agentfrom google.adk.models.lite_llm import LiteLlmfrom google.adk.tools.function_tool import FunctionToolfrom google.adk.tools.tool_context import ToolContextfrom typing import Annotatedasync def store_preferences_tool(    location: Annotated[str, "City or neighborhood the user wants to search in"],    budget: Annotated[int, "Maximum budget in the local currency (e.g., USD)"],    bedrooms: Annotated[int, "Minimum number of bedrooms"],    property_type: Annotated[str, "Type of property (e.g., apartment, house)"],    purpose: Annotated[str, "Purpose of property: buy or rent"],    *,    session,) -> str:    normalized_purpose = purpose.lower()    session.state["user_preferences"] = {        "location": location,        "budget": budget,        "bedrooms": bedrooms,        "property_type": property_type,        "purpose": normalized_purpose,    }    return f"Preferences saved: {session.state['user_preferences']}"store_preferences_tool_wrapped = FunctionTool(store_preferences_tool)preference_agent = Nonetry:    preference_agent = Agent(        model=lite_llm,        name="PreferenceAgent",        instruction="You are the Preference Agent. Your job is to capture and store user preferences using the 'store_preferences_tool'.",        description="Stores user search preferences like location, budget, bedrooms, etc., in session state.",        tools=[store_preferences_tool_wrapped],    )    print(f" Agent '{preference_agent.name}' defined.")except Exception as e:    print(f"Could not define Preference agent. Error: {e}")

 Agent 'PreferenceAgent' defined.


In [5]:
import httpxfrom typing import Optionalfrom google.adk.agents import Agentfrom google.adk.models.lite_llm import LiteLlmfrom google.adk.tools.function_tool import FunctionToolfrom google.adk.runners import Runnerasync def search_listings_tool(*, session) -> str:    preferences = session.state.get("user_preferences")    if not preferences:        return "User preferences not found in session. Please set them first."    api_url = "https://api.rentcast.io/v1/properties/search"    headers = {"Authorization": f"Bearer b610c9d2da81402d92be6e4e6a9affd5"}    params = {        "location": preferences["location"],        "max_price": preferences["budget"],        "min_bedrooms": preferences["bedrooms"],        "property_type": preferences["property_type"],        "limit": 10    }    async with httpx.AsyncClient() as client:        response = await client.get(api_url, headers=headers, params=params)    if response.status_code == 200:        data = response.json()        listings = data.get("results", [])        for listing in listings:            if "link" not in listing:                listing["link"] = f"https://rentcast.io/property/{listing.get('propertyId', 'unknown')}"        if not listings:            return "No listings found matching your preferences."        session.state["search_results"] = listings        return f"{len(listings)} listings found and stored in session."    else:        return f"Failed to fetch listings: {response.status_code} - {response.text}"search_listings_tool_wrapped = FunctionTool(search_listings_tool)search_agent = Nonetry:    search_agent = Agent(        model=lite_llm,        name="SearchAgent",        instruction="""        You are the Search Agent.        Your job is to find real estate listings that match the user's preferences.        Always invoke the tool named `search_listings_tool` to do this.        Do not search or reason by yourself—just call the tool.        """,        description="Searches for real estate listings using stored user preferences.",        tools=[search_listings_tool_wrapped],    )    print(f" Agent '{search_agent.name}' defined.")except Exception as e:    print(f" Could not define Search agent. Error: {e}")

 Agent 'SearchAgent' defined.


In [6]:
from typing import List, Dictfrom google.adk.agents import Agentfrom google.adk.models.lite_llm import LiteLlmfrom google.adk.tools.function_tool import FunctionToolasync def score_listings_tool(*, session) -> str:    listings: List[Dict] = session.state.get("search_results")    preferences: Dict = session.state.get("user_preferences")    if not listings:        return "No search results found. Please perform a search first."    if not preferences:        return "User preferences not found. Please set preferences first."    budget = preferences.get("budget")    preferred_bedrooms = preferences.get("bedrooms")    preferred_property_type = preferences.get("property_type")    def score_listing(listing: Dict) -> float:        score = 0.0        price_diff = abs(budget - listing["price"])        price_score = max(0, (budget - price_diff)) / budget        score += price_score * 50        if listing["bedrooms"] >= preferred_bedrooms:            score += 30        else:            score += (listing["bedrooms"] / preferred_bedrooms) * 30        if listing["property_type"].lower() == preferred_property_type.lower():            score += 20        return score    for listing in listings:        listing["score"] = score_listing(listing)    listings.sort(key=lambda x: x["score"], reverse=True)    top_picks = listings[:3]    session.state["top_picks"] = top_picks    return f"Top {len(top_picks)} properties scored and saved to session."score_listings_tool_wrapped = FunctionTool(score_listings_tool)scoring_agent = Nonetry:    scoring_agent = Agent(        model=lite_llm,        name="ScoringAgent",        instruction="""        You are the Scoring Agent.        Your task is to score property listings based on the user's preferences.        Always invoke the tool named `score_listings_tool` to perform scoring.        Do not write scoring logic yourself.        """,        description="Scores listings based on price, bedrooms, and property type against stored preferences.",        tools=[score_listings_tool_wrapped],    )    print(f" Agent '{scoring_agent.name}' defined.")except Exception as e:    print(f" Could not define Scoring agent. Error: {e}")

 Agent 'ScoringAgent' defined.


In [7]:
from google.adk.agents import Agentfrom google.adk.tools.function_tool import FunctionToolfrom google.adk.models.lite_llm import LiteLlmasync def summarize_top_picks_tool(*, session) -> str:    top_picks = session.state.get("top_picks")    if not top_picks:        return "No top picks found in session. Please run scoring first."    table = "| Title | Price | Location | Type | Bedrooms | Link |\n"    table += "|-------|-------|----------|------|----------|------|\n"    for listing in top_picks:        title = listing.get("title", "N/A")        price = f"${listing.get('price', 'N/A')}"        location = listing.get("location", listing.get("title", "N/A"))  # fallback to title        prop_type = listing.get("property_type", "N/A")        bedrooms = listing.get("bedrooms", "N/A")        link = f"[View]({listing.get('link', '#')})"        table += f"| {title} | {price} | {location} | {prop_type} | {bedrooms} | {link} |\n"    best = top_picks[0]    rec = (        f"\n**Top Recommendation:**\n"        f"The best match is **{best['title']}**, priced at **${best['price']}**, "        f"a {best['property_type']} with **{best['bedrooms']} bedrooms** "        f"located in **{best.get('location', best['title'])}**. "        f"You can view more details [here]({best['link']})."    )    return f"### Top Property Picks\n\n{table}\n{rec}"summarize_top_picks_tool_wrapped = FunctionTool(summarize_top_picks_tool)summary_agent = Nonetry:    summary_agent = Agent(        model=lite_llm,        name="SummaryAgent",        instruction='''        You are the Summary Agent.        Your job is to generate a markdown summary of the top scored real estate listings.        Always call the tool named `summarize_top_picks_tool` to do this.        Do not generate summaries yourself. Let the tool do the work."        ''',        description="Summarizes the top 3 property picks in a markdown table and highlights the best match.",        tools=[summarize_top_picks_tool_wrapped],    )    print(f" Agent '{summary_agent.name}' defined.")except Exception as e:    print(f" Could not define Summary agent. Error: {e}")

 Agent 'SummaryAgent' defined.


In [8]:
from google.adk.tools.function_tool import FunctionToolfrom google.adk.agents import LlmAgentdef agent_as_tool(agent):    async def run_tool(*args, **kwargs):        return await agent.run(*args, **kwargs)    tool = FunctionTool(run_tool)    tool.name = agent.name    return toolpreference_tool = agent_as_tool(preference_agent)search_tool = agent_as_tool(search_agent)scoring_tool = agent_as_tool(scoring_agent)summary_tool = agent_as_tool(summary_agent)root_agent = LlmAgent(    name="RootAgent",    model=lite_llm,     instruction="""You are the Root Agent for a real estate assistant. Follow these steps in order:1. Use the PreferenceAgent to store the user's search preferences.2. Use the SearchAgent to find matching listings.3. Use the ScoringAgent to rank the listings.4. Use the SummaryAgent to summarize the top picks.Do not generate your own responses. Use only the tools provided to you.""",    description="Root agent that delegates tasks to sub-agents in proper order.",    tools=[        preference_tool,        search_tool,        scoring_tool,        summary_tool,    ],)print(f" RootAgent '{root_agent.name}' successfully created.")

 RootAgent 'RootAgent' successfully created.


In [9]:
"""import asyncioimport jsonfrom google.adk.sessions import InMemorySessionServicefrom google.adk.runners import Runnerfrom google.genai import typesasync def run_full_real_estate_flow(root_agent):    session_service = InMemorySessionService()    session_id = "user_session_1"    app_name = "real_estate_assistant"    user_id = "user_123"    session = await session_service.create_session(        session_id=session_id,        app_name=app_name,        user_id=user_id    )    runner = Runner(        agent=root_agent,        app_name=app_name,        session_service=session_service,    )    steps = [        "I want to buy a apartment in San Francisco with a budget of 9000000 USD, at least 3 bedrooms."    ]    for i, user_input in enumerate(steps, start=1):        print(f"\n=== Step {i}: {user_input} ===")        content = types.Content(role='user', parts=[types.Part(text=user_input)])        final_response = None        events = runner.run(user_id=user_id, session_id=session_id, new_message=content)        for event in events:            if event.is_final_response() and event.content and event.content.parts:                final_response = event.content.parts[0].text                print(f"\nAgent Final Response:\n{final_response}")    final_session = await session_service.get_session(        app_name=app_name,        user_id=user_id,        session_id=session_id    )    print("\n=== Final Session State ===")    print(json.dumps(final_session.state, indent=2))await run_full_real_estate_flow(root_agent)"""

'import asyncio\nimport json\nfrom google.adk.sessions import InMemorySessionService\nfrom google.adk.runners import Runner\nfrom google.genai import types\n\nasync def run_full_real_estate_flow(root_agent):\n    session_service = InMemorySessionService()\n    session_id = "user_session_1"\n    app_name = "real_estate_assistant"\n    user_id = "user_123"\n\n    session = await session_service.create_session(\n        session_id=session_id,\n        app_name=app_name,\n        user_id=user_id\n    )\n    runner = Runner(\n        agent=root_agent,\n        app_name=app_name,\n        session_service=session_service,\n\n    )\n\n    steps = [\n        "I want to buy a apartment in San Francisco with a budget of 9000000 USD, at least 3 bedrooms."\n    ]\n\n    for i, user_input in enumerate(steps, start=1):\n        print(f"\n=== Step {i}: {user_input} ===")\n\n        content = types.Content(role=\'user\', parts=[types.Part(text=user_input)])\n\n        final_response = None\n        eve

In [10]:
import gradio as grfrom google.genai import typesfrom google.adk.runners import Runnerrunner = Runner(    agent=root_agent,    app_name=APP_NAME,    session_service=session_service)def query_agent(user_input: str) -> str:    content = types.Content(role="user", parts=[types.Part(text=user_input)])    final_response = None    try:        events = runner.run(            user_id=USER_ID,            session_id=SESSION_ID,            new_message=content        )        for event in events:            if event.is_final_response() and event.content and event.content.parts:                final_response = event.content.parts[0].text                break    except Exception as e:        return f" Error: {e}"    if final_response is None:        return " No final response from the agent."    return final_responsedemo = gr.Interface(    fn=query_agent,    inputs=gr.Textbox(label="Enter your query", placeholder="E.g., I want to rent a 2BHK in Boston under 2000 USD"),    outputs=gr.Textbox(label="AI Assistant Response"),    title=" Real Estate AI Assistant",    description="Chat with the AI to find properties based on your preferences. Ask step-by-step!")demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b225171de4fb3d5f84.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
